Libraries

In [17]:
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from pathlib import Path
import random
import copy

In [18]:
#Function read adjency
def read_col_edge_num_vertices(path):
    edges = []
    num_vertices = 0
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 4 and parts[0] == "p" and parts[1] == "edge":
                num_vertices = parts[2]
            if len(parts) >= 3 and parts[0] == "e":
                v = int(parts[2]) - 1
                u = int(parts[1]) - 1
                
                edges.append((u, v))
                edges.append((v, u))
    return torch.tensor(edges, dtype=torch.long).t().contiguous(), num_vertices


Take a data from folder master_cp/runs of vertices

In [19]:
run_test_path = Path("../master_cp/runs")
instances_path = Path("../tests")

instances = {}

for instance_dir in run_test_path.iterdir():
    if not instance_dir.is_dir():
        continue

    instance_name = instance_dir.name

    csv_files = list(instance_dir.glob("*/vertex_features.csv"))
    
    csv_path = csv_files[0]

    df = pd.read_csv(csv_path)

    if df.empty:
        print("Skip empty:", csv_path)
        continue

    y_label = df["score_contribution"].astype(float)
    time_step = int(df["iteration"].iloc[-1]) + 1
    x_label = df.drop(columns=["score_contribution", "iteration", "vertex_id"])

    X_tensor = torch.tensor(
        x_label.to_numpy(dtype=float),
        dtype=torch.float32
    )  # [N, F]
    Y_tensor = torch.tensor(
        y_label.to_numpy(),
        dtype=torch.float32
    ).view(-1, 1)  # [N, 1]

    col_path = instances_path / f"{instance_name}.col"
    edge_index, num_vertices = read_col_edge_num_vertices(col_path)

    instances[instance_name] = {
        "X": X_tensor,
        "Y": Y_tensor,
        "edge_index": edge_index,
        "num_vertices": num_vertices,
        "time_step": time_step
    }

print(instances)

Skip empty: ..\master_cp\runs\games120\5151f31b-f395-44d4-8ec4-6fe9ef0f3132\vertex_features.csv
Skip empty: ..\master_cp\runs\huck\3639ae6e-979b-4196-87c6-a72482d8adb7\vertex_features.csv
{'1-FullIns_3': {'X': tensor([[ 4.0000,  0.1379,  0.0000,  0.0000, 25.0000, 52.0000],
        [ 4.0000,  0.1379,  0.0000,  0.0000, 25.0000, 52.0000],
        [ 6.0000,  0.2069, -0.0000,  0.0000, 23.0000, 75.0000],
        [ 6.0000,  0.2069,  0.0000,  1.0000, 23.0000, 75.0000],
        [ 6.0000,  0.2069,  0.6667,  2.0000, 23.0000, 41.0000],
        [ 6.0000,  0.2069,  0.0000,  2.0000, 23.0000, 41.0000],
        [ 8.0000,  0.2759,  0.6667,  2.0000, 21.0000, 45.0000],
        [ 8.0000,  0.2759,  0.0000,  2.0000, 21.0000, 22.0000],
        [ 8.0000,  0.2759,  0.6667,  2.0000, 21.0000, 45.0000],
        [ 5.0000,  0.1724,  0.0000,  0.3333, 24.0000, 74.0000],
        [ 5.0000,  0.1724, -0.0000,  0.3333, 24.0000, 74.0000],
        [ 7.0000,  0.2414,  0.0000,  0.3333, 22.0000, 90.0000],
        [ 7.0000,  0.2

Build layer

In [20]:
from torch_geometric.nn import SAGEConv

class GraphSAGE_GRU (nn.Module):
    def __init__(self, input_dim, gnn_hidden_dim=64, gru_hidden_dim=64):
        super().__init__()

        self.conv = SAGEConv(in_channels=input_dim, out_channels=gnn_hidden_dim)
        
        self.gru = nn.GRU(input_size=gnn_hidden_dim, hidden_size=gru_hidden_dim)
        self.linear = nn.Linear(in_features=gru_hidden_dim, out_features=1)

    def forward(self, X, edge_index):
        """
        X_seq: [T, V, F]
        edge_index: [2, E]
        """
        T, V, _ = X.shape

        list = [] # [T, V, hidden_dim]
        for t in range(T):
            h = self.conv(X[t], edge_index) #[V, hidden_dim]
            h = F.relu(h)
            h = F.dropout(h, p=0.2, training=self.training)
            list.append(h)

        L = torch.stack(list, dim=0) #[T, V, hidden_dim]
         
        out, _ = self.gru(L) # out[T, V, gru_hidden_dim]

        return self.linear(out).reshape(T*V, 1) # [T*V, 1]

In [21]:
def split_instances(instances, train=0.7, validation=0.15, test=0.15, seed=42):
    assert abs(train + validation + test - 1.0) < 1e-8

    items = list(instances.items())

    random.seed(seed)
    random.shuffle(items)

    n = len(items)

    n_train = int(train * n)
    n_val = int(validation * n)

    train_items = items[:n_train]
    val_items = items[n_train:n_train + n_val]
    test_items = items[n_train + n_val:]

    return train_items, val_items, test_items

Trainning Function

In [30]:
def train_one_epcho(model, train_data, optimizer, criterion, device) :
    model.train() 

    total_loss = 0
    
    for _, data in train_data:
        optimizer.zero_grad()
        time_step = int(data["time_step"])
        num_v = int(data["num_vertices"])

        X = data["X"].to(device) #[T*V, Features]
        input_dim = X.shape[1]
        X_reshaped = X.view(time_step, num_v, input_dim) # [T, S, Features]

        Y = data["Y"].to(device) #[T*V, 1]

        edge_index = data["edge_index"].to(device) #[2, E]

        output = model(X_reshaped, edge_index)
        
        loss = criterion(output, Y) 
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(train_data)

Testing Function

In [31]:
@torch.no_grad()
def validation (model, val_data, criterion, device):
    model.eval()

    total_loss = 0

    for _, data in val_data:
        time_step = int(data["time_step"])
        num_v = int(data["num_vertices"])

        X = data["X"].to(device) #[T*V, Features]
        input_dim = X.shape[1]
        X_reshaped = X.view(time_step, num_v, input_dim) # [T, S, Features]

        Y = data["Y"].to(device) #[T*V, 1]

        edge_index = data["edge_index"].to(device) #[2, E]

        output = model(X_reshaped, edge_index)
        
        loss = criterion(output, Y) 
    
        total_loss += loss.item()
    return total_loss / len(val_data)

In [32]:
import torch.optim as optim

input_dim = next(iter(instances.values()))["X"].shape[1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data, val_data, test_data = split_instances(
    instances,
    0.7,
    0.15,
    0.15,
    seed=42
)
model = GraphSAGE_GRU(input_dim=input_dim, gnn_hidden_dim=64, gru_hidden_dim=64).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

Trainning and take best model

In [ ]:
epochs = 60

best_val_loss = float("inf")
best_model_state = None
for epoch in range(epochs):
    train_loss = train_one_epcho(model, train_data, optimizer, criterion, device)

    val_loss = validation(model, val_data, criterion, device)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 1/60 | Train Loss: 0.1438 | Val Loss: 0.0346
Epoch 2/60 | Train Loss: 0.0786 | Val Loss: 0.0467
Epoch 3/60 | Train Loss: 0.0612 | Val Loss: 0.0416
Epoch 4/60 | Train Loss: 0.0529 | Val Loss: 0.0387
Epoch 5/60 | Train Loss: 0.0498 | Val Loss: 0.0377
Epoch 6/60 | Train Loss: 0.0458 | Val Loss: 0.0376
Epoch 7/60 | Train Loss: 0.0442 | Val Loss: 0.0397
Epoch 8/60 | Train Loss: 0.0429 | Val Loss: 0.0332
Epoch 9/60 | Train Loss: 0.0432 | Val Loss: 0.0386
Epoch 10/60 | Train Loss: 0.0402 | Val Loss: 0.0324
Epoch 11/60 | Train Loss: 0.0397 | Val Loss: 0.0329
Epoch 12/60 | Train Loss: 0.0399 | Val Loss: 0.0321
Epoch 13/60 | Train Loss: 0.0384 | Val Loss: 0.0321
Epoch 14/60 | Train Loss: 0.0389 | Val Loss: 0.0351
Epoch 15/60 | Train Loss: 0.0377 | Val Loss: 0.0322
Epoch 16/60 | Train Loss: 0.0375 | Val Loss: 0.0336
Epoch 17/60 | Train Loss: 0.0368 | Val Loss: 0.0345
Epoch 18/60 | Train Loss: 0.0374 | Val Loss: 0.0320
Epoch 19/60 | Train Loss: 0.0372 | Val Loss: 0.0390
Epoch 20/60 | Train L

NameError: name 'test_loss' is not defined

Testing

In [35]:
model.load_state_dict(best_model_state)

test_loss = validation(model, test_data, criterion, device)
print(f"Final Test Loss: {test_loss:.4f}")

Final Test Loss: 0.0322


Save model

In [39]:
save_dir = Path('./models')
model_path = save_dir / "sort_vertices.pt"
torch.save(model.state_dict(), model_path)